# Ray Tracing: From Basic Casting to Advanced Rendering

**Author:** Extended from E. Ribeiro's ray-casting framework

This notebook implements a complete recursive ray tracer with the following features:

1. **Basic Ray Tracing** — Recursive ray-surface intersection with depth control
2. **Reflection** — Mirror-like specular reflection via recursive rays
3. **Refraction** — Transparent surfaces using Snell's law and Fresnel approximation
4. **Glossy Reflection** — Perturbed reflection rays for blurred/fuzzy reflections
5. **Multiple Viewpoints** — Rendering the scene from different camera positions
6. **Soft Shadows** — Area light sources approximated by jittered point samples

A feature-flag dictionary (`FEATURES`) lets you toggle each effect independently.

In [ ]:
import numpy as np
from PIL import Image as im
import time
from IPython.display import display

# ──────────────────────────────────────────────────────
# FEATURE FLAGS — toggle each capability on/off
# ──────────────────────────────────────────────────────
FEATURES = {
    'reflection':      True,   # Mirror reflection
    'refraction':      True,   # Transparent/refractive surfaces
    'glossy':          False,  # Glossy (blurred) reflections — slower
    'soft_shadows':    False,  # Area-light soft shadows   — slower
}

MAX_DEPTH = 4           # Recursion depth for reflection/refraction
GLOSSY_SAMPLES = 8      # Rays per glossy reflection
SHADOW_SAMPLES = 8      # Rays per soft-shadow test
GLOSSY_SPREAD  = 0.08   # Angular spread for glossy perturbation

np.random.seed(42)
print("Feature flags:", FEATURES)

## 1. The Ray Equation

A ray is a parametric line in 3-D:

$$\mathbf{p}(t) = \mathbf{e} + \mathbf{d}\,t, \qquad \mathbf{d} = \mathbf{s} - \mathbf{e}$$

where $\mathbf{e}$ is the eye (origin), $\mathbf{s}$ is a point on the image plane, and $t \ge 0$ parameterises distance along the ray.

For each pixel $(i,j)$ in the image we convert to continuous image-plane coordinates $(u,v)$ and shoot a ray from the eye through $(u, v, -f)$ where $f$ is the focal length.

In [ ]:
class Ray:
    """Parametric ray  p(t) = e + d*t,  d = s - e."""

    def __init__(self, e, s):
        self.e = np.array(e, dtype=float)
        self.s = np.array(s, dtype=float)

    @staticmethod
    def from_direction(origin, direction):
        """Construct a Ray from an origin and a direction vector."""
        return Ray(origin, np.array(origin) + np.array(direction))

    def direction(self):
        d = self.s - self.e
        return d / np.linalg.norm(d)

    def get3DPoint(self, t):
        return self.e + (self.s - self.e) * t

## 2. Sphere Intersection

The implicit equation of a sphere with centre $\mathbf{c}$ and radius $r$ is:

$$(\mathbf{p} - \mathbf{c}) \cdot (\mathbf{p} - \mathbf{c}) - r^2 = 0$$

Substituting $\mathbf{p}(t)$ gives the quadratic:

$$\underbrace{(\mathbf{d}\cdot\mathbf{d})}_{A}\;t^2
 + \underbrace{2\,\mathbf{d}\cdot(\mathbf{e}-\mathbf{c})}_{B}\;t
 + \underbrace{(\mathbf{e}-\mathbf{c})\cdot(\mathbf{e}-\mathbf{c}) - r^2}_{C} = 0$$

Solved by the quadratic formula. We keep the smallest positive $t$.

In [ ]:
class Material:
    """Surface material descriptor for Phong shading, reflection, and refraction."""
    def __init__(self, color, ambient=0.15, diffuse=0.7, specular=0.5,
                 shininess=64, reflectivity=0.3, transparency=0.0, ior=1.5):
        self.color       = np.array(color, dtype=float)   # base RGB [0-1]
        self.ambient     = ambient
        self.diffuse     = diffuse
        self.specular    = specular
        self.shininess   = shininess
        self.reflectivity = reflectivity    # 0 = matte, 1 = perfect mirror
        self.transparency = transparency    # 0 = opaque, 1 = fully transparent
        self.ior          = ior             # index of refraction


class Sphere:
    """Sphere with centre, radius, and material."""

    def __init__(self, center, radius, material):
        self.Center = np.array(center, dtype=float)
        self.Radius = float(radius)
        self.material = material

    def Intersect(self, ray):
        d = ray.s - ray.e
        ec = ray.e - self.Center
        A = np.dot(d, d)
        B = 2.0 * np.dot(d, ec)
        C = np.dot(ec, ec) - self.Radius ** 2
        delta = B * B - 4.0 * A * C
        if delta < 0:
            return float('inf')
        sq = np.sqrt(delta)
        t1 = (-B - sq) / (2.0 * A)
        t2 = (-B + sq) / (2.0 * A)
        # Return smallest positive t (epsilon to avoid self-intersection)
        if t1 > 1e-6:
            return t1
        if t2 > 1e-6:
            return t2
        return float('inf')

    def get_normal(self, p):
        n = p - self.Center
        return n / np.linalg.norm(n)

## 3. Plane Intersection

A plane through point $\mathbf{p}_1$ with unit normal $\mathbf{n}$:

$$(\mathbf{p} - \mathbf{p}_1) \cdot \mathbf{n} = 0$$

Substituting the ray equation and solving for $t$:

$$t = \frac{(\mathbf{p}_1 - \mathbf{e}) \cdot \mathbf{n}}{\mathbf{d} \cdot \mathbf{n}}$$

If $\mathbf{d}\cdot\mathbf{n} \approx 0$ the ray is parallel to the plane (no intersection). We only accept $t > 0$ (intersection in front of the eye).

### Checkerboard Pattern

To create a checkerboard we use the floor of the $x$ and $z$ coordinates of the hit point. If the sum of these integer coordinates is even, we use the primary material; otherwise the secondary material.

In [ ]:
class Plane:
    """Infinite plane defined by a point and a normal, with optional checkerboard."""

    def __init__(self, p1, n, material, material2=None, check_scale=100.0):
        self.p1 = np.array(p1, dtype=float)
        self.n  = np.array(n, dtype=float)
        self.n  = self.n / np.linalg.norm(self.n)
        self.material  = material        # primary material
        self.material2 = material2       # secondary material (checkerboard)
        self.check_scale = check_scale

    def Intersect(self, ray):
        d = ray.s - ray.e
        denom = np.dot(d, self.n)
        if abs(denom) < 1e-8:
            return float('inf')        # Ray is parallel to plane
        t = np.dot(self.p1 - ray.e, self.n) / denom
        if t < 1e-6:
            return float('inf')        # Intersection behind eye
        return t

    def get_normal(self, p):
        return self.n

    def get_material_at(self, p):
        """Return material — supports checkerboard pattern."""
        if self.material2 is None:
            return self.material
        s = self.check_scale
        if (int(np.floor(p[0] / s)) + int(np.floor(p[2] / s))) % 2 == 0:
            return self.material
        return self.material2

## 4. Phong Reflection Model (Blinn-Phong Variant)

The colour at a surface point is:

$$I = k_a \, C + k_d \, C \max(0, \mathbf{n}\cdot\mathbf{l}) + k_s \max(0, \mathbf{n}\cdot\mathbf{h})^{\alpha}$$

where:
- $k_a, k_d, k_s$ are the ambient, diffuse, and specular coefficients
- $C$ is the surface colour, $\mathbf{n}$ the surface normal, $\mathbf{l}$ the unit vector toward the light
- $\mathbf{h} = \frac{\mathbf{l}+\mathbf{v}}{\|\mathbf{l}+\mathbf{v}\|}$ is the half vector (Blinn-Phong)
- $\alpha$ is the shininess exponent

## 5. Recursive Reflection

$$\mathbf{r} = \mathbf{d} - 2(\mathbf{d}\cdot\mathbf{n})\,\mathbf{n}$$

The reflected ray is traced recursively up to `MAX_DEPTH`. The reflectivity coefficient $\rho$ blends:

$$I_{\text{final}} = (1-\rho)\,I_{\text{local}} + \rho\,I_{\text{reflected}}$$

## 6. Refraction (Snell's Law)

Given incident direction $\mathbf{d}$ (unit), normal $\mathbf{n}$, and ratio $\eta = n_1 / n_2$:

$$\cos\theta_t = \sqrt{1 - \eta^2(1-\cos^2\theta_i)}$$
$$\mathbf{t} = \eta\,\mathbf{d} + (\eta\cos\theta_i - \cos\theta_t)\,\mathbf{n}$$

Total internal reflection occurs when the term under the square root is negative.

### Fresnel Approximation (Schlick)

$$R(\theta) = R_0 + (1-R_0)(1-\cos\theta)^5, \quad R_0 = \left(\frac{n_1-n_2}{n_1+n_2}\right)^2$$

This blends reflection and refraction energy realistically — more reflection at grazing angles.

## 7. Glossy Reflection

Instead of a single mirror ray, we jitter the reflection direction by sampling a small cone around $\mathbf{r}$:

$$\mathbf{r}' = \text{normalize}\!\left(\mathbf{r} + \sigma\,(\cos\phi\;\mathbf{t}_1 + \sin\phi\;\mathbf{t}_2)\right)$$

Each perturbed ray is traced and the results averaged, producing a blurred reflection.

## 8. Soft Shadows

A point light creates hard shadows. To simulate an area light, we jitter the light position within a sphere and cast $N$ shadow rays:

$$S = \frac{1}{N}\sum_{k=1}^{N}\text{visible}_k$$

The fraction that reach the light determines the shadow softness (penumbra).

In [ ]:
class Camera:
    """Pinhole camera that can be positioned and oriented anywhere in 3-D.

    Builds an orthonormal basis {u, v, w} from eye, look_at, and up vectors,
    enabling arbitrary camera placement for multiple viewpoints.
    """
    nchannels = 3

    def __init__(self, eye, look_at, up, f, nrows, ncols):
        self.eye   = np.array(eye, dtype=float)
        self.f     = f
        self.nrows = nrows
        self.ncols = ncols
        self.I     = np.zeros([nrows, ncols, self.nchannels])

        # Build orthonormal basis  (w points away from look_at)
        self.w = self.eye - np.array(look_at, dtype=float)
        self.w = self.w / np.linalg.norm(self.w)
        self.u = np.cross(np.array(up, dtype=float), self.w)
        self.u = self.u / np.linalg.norm(self.u)
        self.v = np.cross(self.w, self.u)

    def ij2uv(self, i, j):
        u_coord =  (j + 0.5) - self.ncols / 2.0
        v_coord = -(i + 0.5) + self.nrows / 2.0
        return u_coord, v_coord

    def constructRayThroughPixel(self, i, j):
        uc, vc = self.ij2uv(i, j)
        direction = uc * self.u + vc * self.v - self.f * self.w
        s = self.eye + direction
        return Ray(self.eye, s)


class PointLight:
    """Point (or area) light source."""
    def __init__(self, position, color=np.ones(3), intensity=1.0, radius=0.0):
        self.position  = np.array(position, dtype=float)
        self.color     = np.array(color, dtype=float)
        self.intensity = intensity
        self.radius    = radius   # >0 makes it an area light for soft shadows


class HitInfo:
    """Stores intersection data: which object was hit, where, and at what t."""
    def __init__(self, obj, point, t):
        self.obj = obj
        self.p   = point
        self.t   = t

## Core Ray Tracer — Recursive Implementation

The `trace_ray` function is the heart of the renderer. For each ray it:

1. Finds the closest intersection with any scene object
2. Computes local Blinn-Phong illumination (with shadow test)
3. If `reflection` is enabled, recursively traces the reflected ray
4. If `refraction` is enabled, computes Snell's law refraction and blends with Fresnel
5. If `glossy` is enabled, uses multiple jittered reflection rays
6. If `soft_shadows` is enabled, casts multiple shadow rays to the jittered light

In [ ]:
def reflect(d, n):
    """Reflect direction d about normal n."""
    return d - 2.0 * np.dot(d, n) * n


def refract(d, n, eta):
    """Compute refracted direction using Snell's law.
    Returns None on total internal reflection.
    """
    cos_i = -np.dot(d, n)
    sin2_t = eta * eta * (1.0 - cos_i * cos_i)
    if sin2_t > 1.0:
        return None   # Total internal reflection
    cos_t = np.sqrt(1.0 - sin2_t)
    return eta * d + (eta * cos_i - cos_t) * n


def schlick(cos_theta, n1, n2):
    """Schlick's Fresnel approximation."""
    r0 = ((n1 - n2) / (n1 + n2)) ** 2
    return r0 + (1.0 - r0) * (1.0 - cos_theta) ** 5


def perturb_direction(d, spread):
    """Perturb unit direction d within a cone of half-angle ~spread (radians).
    Used for glossy reflections.
    """
    d = d / np.linalg.norm(d)
    # Build local tangent frame
    helper = np.array([1, 0, 0], dtype=float) if abs(d[0]) < 0.9 else np.array([0, 1, 0], dtype=float)
    t1 = np.cross(d, helper)
    t1 /= np.linalg.norm(t1)
    t2 = np.cross(d, t1)
    # Random offset in tangent disk
    angle  = 2.0 * np.pi * np.random.random()
    radius = spread * np.random.random()
    perturbed = d + radius * (np.cos(angle) * t1 + np.sin(angle) * t2)
    return perturbed / np.linalg.norm(perturbed)


def find_closest_hit(scene_objects, ray):
    """Find the closest intersection of ray with any scene object."""
    closest_t = float('inf')
    closest_hit = None
    for obj in scene_objects:
        t = obj.Intersect(ray)
        if 1e-5 < t < closest_t:
            closest_t = t
            closest_hit = HitInfo(obj, ray.get3DPoint(t), t)
    return closest_hit


def shadow_factor(scene_objects, point, light, features):
    """Compute shadow factor [0,1]. 0 = fully shadowed, 1 = fully lit.
    With soft_shadows enabled and light.radius > 0, jitters the light position.
    """
    if features.get('soft_shadows') and light.radius > 0:
        lit = 0
        for _ in range(SHADOW_SAMPLES):
            offset = np.random.randn(3) * light.radius
            jittered_pos = light.position + offset
            to_light = jittered_pos - point
            dist = np.linalg.norm(to_light)
            shadow_ray = Ray.from_direction(point, to_light / dist)
            hit = find_closest_hit(scene_objects, shadow_ray)
            if hit is None or hit.t > dist:
                lit += 1
        return lit / SHADOW_SAMPLES
    else:
        to_light = light.position - point
        dist = np.linalg.norm(to_light)
        shadow_ray = Ray.from_direction(point, to_light / dist)
        hit = find_closest_hit(scene_objects, shadow_ray)
        return 1.0 if (hit is None or hit.t > dist) else 0.0


def trace_ray(ray, scene_objects, lights, features, depth=0):
    """Recursive ray tracer. Returns RGB colour as numpy array [0-1]."""
    if depth > MAX_DEPTH:
        return np.zeros(3)

    hit = find_closest_hit(scene_objects, ray)
    if hit is None:
        # Sky gradient background
        d = ray.direction()
        t_sky = 0.5 * (d[1] + 1.0)
        return (1.0 - t_sky) * np.array([1.0, 1.0, 1.0]) + t_sky * np.array([0.5, 0.7, 1.0])

    obj = hit.obj
    p   = hit.p
    n   = obj.get_normal(p)

    # Get material (planes may have checkerboard)
    if hasattr(obj, 'get_material_at'):
        mat = obj.get_material_at(p)
    else:
        mat = obj.material

    # Ensure normal faces the incoming ray
    d_dir = ray.direction()
    if np.dot(n, d_dir) > 0:
        n = -n

    # ── Local illumination (Blinn-Phong) ──
    color = mat.ambient * mat.color   # ambient term

    for light in lights:
        sf = shadow_factor(scene_objects, p + n * 1e-4, light, features)
        if sf <= 0:
            continue

        l = light.position - p
        l = l / np.linalg.norm(l)

        # Diffuse
        diff = max(0.0, np.dot(n, l))
        color += sf * mat.diffuse * mat.color * diff * light.color * light.intensity

        # Specular (Blinn-Phong half-vector)
        v = -d_dir
        h = l + v
        h = h / np.linalg.norm(h)
        spec = max(0.0, np.dot(n, h)) ** mat.shininess
        color += sf * mat.specular * spec * light.color * light.intensity

    # ── Reflection ──
    if features.get('reflection') and mat.reflectivity > 0 and depth < MAX_DEPTH:
        r_dir = reflect(d_dir, n)

        if features.get('glossy') and GLOSSY_SAMPLES > 1:
            # Glossy: average multiple perturbed reflection rays
            refl_color = np.zeros(3)
            for _ in range(GLOSSY_SAMPLES):
                pr = perturb_direction(r_dir, GLOSSY_SPREAD)
                refl_ray = Ray.from_direction(p + n * 1e-4, pr)
                refl_color += trace_ray(refl_ray, scene_objects, lights, features, depth + 1)
            refl_color /= GLOSSY_SAMPLES
        else:
            refl_ray = Ray.from_direction(p + n * 1e-4, r_dir)
            refl_color = trace_ray(refl_ray, scene_objects, lights, features, depth + 1)

        color = color * (1.0 - mat.reflectivity) + refl_color * mat.reflectivity

    # ── Refraction ──
    if features.get('refraction') and mat.transparency > 0 and depth < MAX_DEPTH:
        # Determine if entering or exiting the medium
        entering = np.dot(ray.direction(), obj.get_normal(p)) < 0
        if entering:
            n_refr = n
            eta = 1.0 / mat.ior
            n1, n2 = 1.0, mat.ior
        else:
            n_refr = -n
            eta = mat.ior
            n1, n2 = mat.ior, 1.0

        cos_i = abs(np.dot(d_dir, n_refr))
        fresnel = schlick(cos_i, n1, n2)

        refr_dir = refract(d_dir, n_refr, eta)
        if refr_dir is not None:
            refr_ray = Ray.from_direction(p - n_refr * 1e-4, refr_dir)
            refr_color = trace_ray(refr_ray, scene_objects, lights, features, depth + 1)
            # Blend using Fresnel: more reflection at grazing angles
            color = color * fresnel + refr_color * (1.0 - fresnel) * mat.transparency

    return np.clip(color, 0, 1)

## Scene Definition

The scene contains:
- A **red opaque sphere** (reflective, Phong-shaded)
- A **glass sphere** (transparent with IoR = 1.52, slight reflectivity)
- A **checkerboard ground plane**
- Two lights (one white, one blue-tinted fill light)

In [ ]:
def build_scene():
    """Create the scene: two spheres + a ground plane + lights."""
    objects = []

    # Red sphere (reflective, opaque)
    mat_red = Material([0.9, 0.1, 0.1], ambient=0.1, diffuse=0.7,
                       specular=0.6, shininess=80, reflectivity=0.3)
    objects.append(Sphere([-100, 0, -600], 80, mat_red))

    # Glass sphere (transparent + slightly reflective)
    mat_glass = Material([0.95, 0.95, 1.0], ambient=0.02, diffuse=0.1,
                         specular=0.9, shininess=200, reflectivity=0.1,
                         transparency=0.85, ior=1.52)
    objects.append(Sphere([100, -10, -400], 70, mat_glass))

    # Ground plane — checkerboard
    mat_floor1 = Material([0.9, 0.9, 0.9], ambient=0.15, diffuse=0.6,
                          specular=0.2, shininess=16, reflectivity=0.2)
    mat_floor2 = Material([0.15, 0.15, 0.15], ambient=0.15, diffuse=0.6,
                          specular=0.2, shininess=16, reflectivity=0.2)
    objects.append(Plane([0, -80, 0], [0, 1, 0], mat_floor1, mat_floor2, check_scale=100))

    return objects


def build_lights(area=False):
    """Create lights. If area=True, uses non-zero radius for soft shadows."""
    r = 50 if area else 0
    r2 = 30 if area else 0
    return [
        PointLight([200, 300, 100],  color=[1, 1, 1],       intensity=1.0, radius=r),
        PointLight([-300, 200, 50],  color=[0.4, 0.4, 0.6], intensity=0.5, radius=r2),
    ]


def render(camera, scene_objects, lights, features, label=""):
    """Render the scene pixel by pixel and return a PIL Image."""
    nrows, ncols = camera.nrows, camera.ncols
    t0 = time.time()

    for i in range(nrows):
        if i % 64 == 0:
            pct = 100.0 * i / nrows
            print(f"\r  Rendering{' '+label if label else ''}: {pct:5.1f}%", end='', flush=True)
        for j in range(ncols):
            ray = camera.constructRayThroughPixel(i, j)
            color = trace_ray(ray, scene_objects, lights, features)
            camera.I[i, j, :] = color * 255.0

    elapsed = time.time() - t0
    print(f"\r  Rendering{' '+label if label else ''}: 100.0%  ({elapsed:.1f}s)")

    return im.fromarray(np.uint8(np.clip(camera.I, 0, 255)))


objects = build_scene()
print("Scene built:", len(objects), "objects")

## Result 1: Basic Ray Tracing (Phong Shading Only)

With all advanced features disabled, we see the Blinn-Phong local illumination model: ambient + diffuse + specular components, hard shadows, and the checkerboard ground plane. No reflection or refraction — surfaces show only their local colour.

In [ ]:
features_basic = {'reflection': False, 'refraction': False, 'glossy': False, 'soft_shadows': False}

cam1 = Camera(eye=[0, 50, 200], look_at=[0, 0, -400], up=[0, 1, 0],
              f=250, nrows=256, ncols=256)
lights_hard = build_lights(area=False)

img_basic = render(cam1, objects, lights_hard, features_basic, label="basic")
display(img_basic)
img_basic.save('01_basic_phong.png')

## Result 2: Reflection

Enabling mirror reflection, rays that hit a surface bounce according to:

$$\mathbf{r} = \mathbf{d} - 2(\mathbf{d}\cdot\mathbf{n})\,\mathbf{n}$$

The reflected ray is traced recursively up to `MAX_DEPTH`. The reflectivity coefficient blends local colour with reflected colour. Notice how the checkerboard and red sphere appear reflected in the glass sphere, and the floor reflects both spheres.

In [ ]:
features_refl = {'reflection': True, 'refraction': False, 'glossy': False, 'soft_shadows': False}

cam2 = Camera(eye=[0, 50, 200], look_at=[0, 0, -400], up=[0, 1, 0],
              f=250, nrows=256, ncols=256)

img_refl = render(cam2, objects, lights_hard, features_refl, label="reflection")
display(img_refl)
img_refl.save('02_reflection.png')

## Result 3: Refraction

The glass sphere (IoR = 1.52) refracts light through it using Snell's law:

$$\eta = \frac{n_1}{n_2}, \quad \cos\theta_t = \sqrt{1 - \eta^2(1-\cos^2\theta_i)}$$

$$\mathbf{t} = \eta\,\mathbf{d} + (\eta\cos\theta_i - \cos\theta_t)\,\mathbf{n}$$

Fresnel blending (Schlick approximation) controls the reflection-to-refraction ratio. Observe how the background is distorted through the glass sphere and the edges show stronger reflection (Fresnel effect).

In [ ]:
features_refr = {'reflection': True, 'refraction': True, 'glossy': False, 'soft_shadows': False}

cam3 = Camera(eye=[0, 50, 200], look_at=[0, 0, -400], up=[0, 1, 0],
              f=250, nrows=256, ncols=256)

img_refr = render(cam3, objects, lights_hard, features_refr, label="reflection+refraction")
display(img_refr)
img_refr.save('03_refraction.png')

## Result 4: Glossy Reflection

Instead of tracing a single perfect mirror ray, we sample multiple rays in a cone around the reflection direction. Each perturbed direction is:

$$\mathbf{r}' = \text{normalize}\!\left(\mathbf{r} + \sigma\,(\cos\phi\;\mathbf{t}_1 + \sin\phi\;\mathbf{t}_2)\right)$$

where $\sigma$ = `GLOSSY_SPREAD` controls the blur and $\{\mathbf{t}_1, \mathbf{t}_2\}$ span the plane perpendicular to $\mathbf{r}$. Averaging the traced colours produces a blurred/glossy reflection.

In [ ]:
features_glossy = {'reflection': True, 'refraction': True, 'glossy': True, 'soft_shadows': False}

GLOSSY_SAMPLES = 6   # Reduce for speed

cam4 = Camera(eye=[0, 50, 200], look_at=[0, 0, -400], up=[0, 1, 0],
              f=250, nrows=256, ncols=256)

img_glossy = render(cam4, objects, lights_hard, features_glossy, label="glossy")
display(img_glossy)
img_glossy.save('04_glossy.png')

GLOSSY_SAMPLES = 8   # Restore default

## Result 5: Multiple Viewpoints

The camera can be positioned anywhere in 3-D space. The orthonormal basis $\{\mathbf{u}, \mathbf{v}, \mathbf{w}\}$ is computed from `eye`, `look_at`, and `up` vectors, allowing arbitrary camera placement and orientation. Here we show three views: front, side, and top-down.

In [ ]:
features_views = {'reflection': True, 'refraction': True, 'glossy': False, 'soft_shadows': False}

viewpoints = [
    {"eye": [0, 50, 200],    "look_at": [0, 0, -400], "label": "Front"},
    {"eye": [350, 80, -200], "look_at": [0, 0, -500], "label": "Side"},
    {"eye": [0, 350, 0],    "look_at": [0, 0, -500],  "label": "Top-down"},
]

view_images = []
for vp in viewpoints:
    cam_v = Camera(eye=vp['eye'], look_at=vp['look_at'], up=[0, 1, 0],
                   f=250, nrows=200, ncols=200)
    img_v = render(cam_v, objects, lights_hard, features_views, label=vp['label'])
    view_images.append(img_v)

# Combine into a side-by-side strip
total_w = sum(img_.width for img_ in view_images) + 20 * (len(view_images) - 1)
strip = im.new('RGB', (total_w, view_images[0].height), (40, 40, 40))
x_off = 0
for img_v in view_images:
    strip.paste(img_v, (x_off, 0))
    x_off += img_v.width + 20

display(strip)
strip.save('05_viewpoints.png')
print("Views:", [vp['label'] for vp in viewpoints])

## Result 6: Soft Shadows

A point light produces infinitely sharp shadow boundaries. To simulate a spatially extended (area) light, we jitter the light position uniformly within a sphere of radius $R_{\text{light}}$ and cast $N$ shadow rays. The shadow factor becomes:

$$S = \frac{1}{N}\sum_{k=1}^{N}\text{visible}_k$$

This creates soft penumbra at shadow edges, where only a fraction of the area light is visible from the surface point. Compare the shadow edges here with the hard shadows in Result 1.

In [ ]:
features_soft = {'reflection': True, 'refraction': True, 'glossy': False, 'soft_shadows': True}

SHADOW_SAMPLES = 12

lights_area = build_lights(area=True)

cam6 = Camera(eye=[0, 50, 200], look_at=[0, 0, -400], up=[0, 1, 0],
              f=250, nrows=256, ncols=256)

img_soft = render(cam6, objects, lights_area, features_soft, label="soft shadows")
display(img_soft)
img_soft.save('06_soft_shadows.png')

SHADOW_SAMPLES = 8   # Restore

## Final: All Features Combined

All effects enabled simultaneously: Blinn-Phong shading, recursive reflection and refraction with Fresnel, glossy reflections, and soft shadows.

In [ ]:
features_all = {
    'reflection':   True,
    'refraction':   True,
    'glossy':       True,
    'soft_shadows': True,
}

GLOSSY_SAMPLES = 4
SHADOW_SAMPLES = 8

lights_all = build_lights(area=True)

cam_all = Camera(eye=[0, 50, 200], look_at=[0, 0, -400], up=[0, 1, 0],
                 f=250, nrows=256, ncols=256)

img_all = render(cam_all, objects, lights_all, features_all, label="ALL features")
display(img_all)
img_all.save('07_all_features.png')

print("\nAll renders complete!")

## Summary

| # | Feature | Technique | Toggle |
|---|---------|-----------|--------|
| 1 | Basic ray tracing | Recursive ray-surface intersection (sphere + plane), Blinn-Phong shading | Always on |
| 2 | Reflection | Mirror rays via $\mathbf{r}=\mathbf{d}-2(\mathbf{d}\cdot\mathbf{n})\mathbf{n}$, recursive to `MAX_DEPTH` | `reflection` |
| 3 | Refraction | Snell's law + Schlick Fresnel approximation | `refraction` |
| 4 | Glossy reflection | Monte Carlo sampling of perturbed reflection rays | `glossy` |
| 5 | Multiple viewpoints | Arbitrary camera with orthonormal basis | Change `eye`/`look_at` |
| 6 | Soft shadows | Jittered area-light shadow rays | `soft_shadows` |

Toggle any feature by setting it to `True`/`False` in the `FEATURES` dictionary or by passing a custom dict to `render()`.